In [ ]:
#FL TESTING ON MAIN DATA WITHOUT DP
"""
BastionFed - Federated Learning Training (Kaggle Version)
=========================================
- 4 clients, splitting local data 70-15-15
- Warm start from Phase 1 pretrained weights
- Trains DNN & ResNet on Client Train (70%)
- Trains Meta-Fusion (PyTorch LR) on Client Val (15%)
- Evaluates Local Models on Client Test sets (15%)
- FedAvg aggregation across all 3 models (DNN, ResNet, Meta-Fusion)
- Evaluates Global Model on MAIN Dataset Test set (15%)
"""

import os
import copy
import json
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from PIL import Image
from tqdm.auto import tqdm

# ==========================================
# KAGGLE CONFIGURATION
# ==========================================
KAGGLE_INPUT = Path("/kaggle/input/datasets/hammadali0459/final-dataset") 
OUTPUT_DIR   = Path("/kaggle/working/_fl_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Define where client data lives inside the Kaggle input
CLIENT_IMG_DIRS = {
    1: (Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_1/client_1/malware"), Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_1/client_1/benign")),
    2: (Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_2/client_2/malware"), Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_2/client_2/benign")),
    3: (Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_3/malware"), Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_3/benign")),
    4: (Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_4/malware"), Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_4/benign")),
}

CLIENT_CSVS = {
    1: Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_1_data.csv"),
    2: Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_2_data.csv"),
    3: Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_3_data.csv"),
    4: Path("/kaggle/input/datasets/hammadali0459/final-dataset/client_4_data.csv"),
}

# --- NEW: MAIN DATA CONFIGURATION ---
MAIN_CSV       = Path("/kaggle/input/datasets/hammadali0459/final-dataset/main_data.csv")
MAIN_IMG_DIRS  = (Path("/kaggle/input/datasets/hammadali0459/final-dataset/mian_image/malware"), Path("/kaggle/input/datasets/hammadali0459/final-dataset/mian_image/benign"))

DNN_WEIGHTS    = Path("/kaggle/input/datasets/hammadali0459/weights/dnn weights")
RESNET_WEIGHTS = Path("/kaggle/input/datasets/hammadali0459/weights/resnet eights") 

FL_ROUNDS    = 10
LOCAL_EPOCHS = 20
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 1e-4

TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ==========================================
# MODEL DEFINITIONS
# ==========================================
class DeepDNN(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.BatchNorm1d(1024), nn.GELU(), nn.Dropout(0.45),
            nn.Linear(1024, 512),  nn.BatchNorm1d(512),  nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 256),   nn.BatchNorm1d(256),   nn.GELU(), nn.Dropout(0.35),
            nn.Linear(256, 128),   nn.BatchNorm1d(128),   nn.GELU(), nn.Dropout(0.25),
            nn.Linear(128, 1),
        )
    def forward(self, x):
        return self.net(x)

def build_resnet() -> nn.Module:
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m

class MetaFusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 1)
    def forward(self, x):
        return torch.sigmoid(self.fc(x)).squeeze(1)

# ==========================================
# DATA LOADING & SPLITTING
# ==========================================
def build_image_index(malware_dir: Path, benign_dir: Path) -> Dict[str, Tuple[str, int]]:
    idx = {}
    if malware_dir.exists():
        for f in malware_dir.rglob("*.png"):
            idx[f.stem.split("_")[0].strip().lower()] = (str(f), 1)
    if benign_dir.exists():
        for f in benign_dir.rglob("*.png"):
            idx[f.stem.split("_")[0].strip().lower()] = (str(f), 0)
    return idx

def parse_label(raw) -> int:
    s = str(raw).strip().lower()
    return 1 if s in ("malware", "1", "1.0") else 0

class ClientDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feat_map: Dict[str, np.ndarray], img_index: Dict[str, Tuple[str, int]]):
        self.rows = []
        for sha, label in zip(df["sha256"], df["label"]):
            sha = str(sha).strip().lower()
            if sha in feat_map and sha in img_index:
                self.rows.append((feat_map[sha], img_index[sha][0], parse_label(label)))

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        fv, img_path, label = self.rows[idx]
        img = TRANSFORM(Image.open(img_path).convert("RGB"))
        return torch.tensor(fv, dtype=torch.float32), img, torch.tensor(label, dtype=torch.long)

def prep_client_data(csv_path: Path, img_index: dict, scaler: StandardScaler):
    df = pd.read_csv(csv_path)
    df['sha256'] = df['sha256'].astype(str).str.strip().str.lower()
    
    csv_hashes = set(df['sha256'])
    img_hashes = set(img_index.keys())
    valid_shas = csv_hashes.intersection(img_hashes)
    
    print(f"    -> Found {len(valid_shas)} matched Image+CSV pairs out of {len(df)} CSV rows for {csv_path.name}.")
    
    if len(valid_shas) == 0:
        raise ValueError(f"CRITICAL: 0 matching hashes for {csv_path.name}!")
        
    df = df[df['sha256'].isin(valid_shas)].copy()
    
    X_feats = df.select_dtypes(include=[np.number]).values.astype(np.float32)
    X_scaled = scaler.transform(X_feats)
    feat_map = {sha: f for sha, f in zip(df["sha256"], X_scaled)}
    
    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])
    
    return (ClientDataset(train_df, feat_map, img_index),
            ClientDataset(val_df, feat_map, img_index),
            ClientDataset(test_df, feat_map, img_index))

# ==========================================
# LOCAL CLIENT TRAINING (ALL 3 MODELS)
# ==========================================
def local_train(dnn, img_model, meta_model, train_loader, val_loader, epochs):
    dnn.train(); img_model.train()
    opt_uni = optim.Adam(list(dnn.parameters()) + list(img_model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
    
    for epoch in range(epochs):
        for fv, img, labels in train_loader:
            fv, img, labels = fv.to(DEVICE), img.to(DEVICE), labels.to(DEVICE)
            opt_uni.zero_grad()
            loss = F.binary_cross_entropy((torch.sigmoid(dnn(fv).squeeze(1)) + F.softmax(img_model(img), dim=1)[:, 1]) / 2.0, labels.float())
            loss.backward()
            opt_uni.step()

    dnn.eval(); img_model.eval(); meta_model.train()
    opt_meta = optim.Adam(meta_model.parameters(), lr=1e-3)
    
    for epoch in range(epochs): 
        for fv, img, labels in val_loader:
            fv, img, labels = fv.to(DEVICE), img.to(DEVICE), labels.to(DEVICE)
            with torch.no_grad():
                d_prob = torch.sigmoid(dnn(fv).squeeze(1))
                i_prob = F.softmax(img_model(img), dim=1)[:, 1]
            
            meta_input = torch.stack([d_prob, i_prob], dim=1)
            opt_meta.zero_grad()
            meta_loss = F.binary_cross_entropy(meta_model(meta_input), labels.float())
            meta_loss.backward()
            opt_meta.step()

    return ({k: v.cpu() for k, v in dnn.state_dict().items()},
            {k: v.cpu() for k, v in img_model.state_dict().items()},
            {k: v.cpu() for k, v in meta_model.state_dict().items()})

# ==========================================
# FEDAVG
# ==========================================
def fedavg(state_dicts: List[dict]) -> dict:
    avg = copy.deepcopy(state_dicts[0])
    for key in avg:
        avg[key] = torch.stack([sd[key].float() for sd in state_dicts], dim=0).mean(dim=0)
    return avg

# ==========================================
# EVALUATION FUNCTION (RENAMED to handle both local and global)
# ==========================================
@torch.no_grad()
def evaluate_model(dnn, img_model, meta_model, test_loaders):
    dnn.eval(); img_model.eval(); meta_model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    for cid, loader in test_loaders.items():
        for fv, img, labels in loader:
            fv, img, labels = fv.to(DEVICE), img.to(DEVICE), labels.to(DEVICE)
            
            d_prob = torch.sigmoid(dnn(fv).squeeze(1))
            i_prob = F.softmax(img_model(img), dim=1)[:, 1]
            
            meta_input = torch.stack([d_prob, i_prob], dim=1)
            final_prob = meta_model(meta_input)
            
            preds = (final_prob >= 0.5).int()
            
            all_probs.extend(final_prob.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.5 
    return acc, f1, rec, auc

# ==========================================
# MAIN EXECUTION
# ==========================================
def main():
    print("Loading Pretrained Weights...")
    global_dnn_sd = torch.load(DNN_WEIGHTS, map_location="cpu", weights_only=False)
    global_img_sd = torch.load(RESNET_WEIGHTS, map_location="cpu", weights_only=False)
    
    global_meta_model = MetaFusionNet()
    global_meta_sd = global_meta_model.state_dict()

    # Setup Scaler
    main_df = pd.read_csv(MAIN_CSV)
    main_scaler = StandardScaler().fit(main_df.select_dtypes(include=[np.number]).values.astype(np.float32))
    n_features = len(main_scaler.mean_)

    # 1. Prep Main Data for Global Evaluation
    print("Prepping Main Global Data (Extracting 15% Test Set)...")
    main_img_idx = build_image_index(*MAIN_IMG_DIRS)
    _, _, main_test_ds = prep_client_data(MAIN_CSV, main_img_idx, main_scaler)
    main_test_loader = DataLoader(main_test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # 2. Prep Client Data
    print("Prepping Client Data (70/15/15 Splits)...")
    client_train, client_val, client_test = {}, {}, {}
    for cid in range(1, 5):
        img_idx = build_image_index(*CLIENT_IMG_DIRS[cid])
        tr_ds, vl_ds, ts_ds = prep_client_data(CLIENT_CSVS[cid], img_idx, main_scaler)
        client_train[cid] = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, drop_last=True)
        client_val[cid] = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False)
        client_test[cid] = DataLoader(ts_ds, batch_size=BATCH_SIZE, shuffle=False)

    print("\nStarting FL Rounds...")
    for fl_round in range(1, FL_ROUNDS + 1):
        print(f"\n{'='*40}\n  ROUND {fl_round}\n{'='*40}")
        dnn_updates, img_updates, meta_updates = [], [], []

        for cid in range(1, 5):
            print(f"  Training Client {cid}...")
            dnn = DeepDNN(n_features).to(DEVICE)
            img_model = build_resnet().to(DEVICE)
            meta = MetaFusionNet().to(DEVICE)
            
            dnn.load_state_dict(global_dnn_sd)
            img_model.load_state_dict(global_img_sd)
            meta.load_state_dict(global_meta_sd)

            new_dnn, new_img, new_meta = local_train(dnn, img_model, meta, client_train[cid], client_val[cid], LOCAL_EPOCHS)
            dnn_updates.append(new_dnn); img_updates.append(new_img); meta_updates.append(new_meta)

            # --- NEW: Evaluate Local Model on Client's Test Set ---
            eval_local_dnn = DeepDNN(n_features).to(DEVICE)
            eval_local_img = build_resnet().to(DEVICE)
            eval_local_meta = MetaFusionNet().to(DEVICE)
            
            eval_local_dnn.load_state_dict(new_dnn)
            eval_local_img.load_state_dict(new_img)
            eval_local_meta.load_state_dict(new_meta)
            
            c_acc, c_f1, c_rec, c_auc = evaluate_model(eval_local_dnn, eval_local_img, eval_local_meta, {cid: client_test[cid]})
            print(f"    -> Local Test Acc: {c_acc:.4f} | F1: {c_f1:.4f} | AUC: {c_auc:.4f}")

        # Aggregation
        global_dnn_sd = fedavg(dnn_updates)
        global_img_sd = fedavg(img_updates)
        global_meta_sd = fedavg(meta_updates)
        
        # --- NEW: Evaluate Global Model on MAIN Test Set ---
        eval_global_dnn = DeepDNN(n_features).to(DEVICE)
        eval_global_img = build_resnet().to(DEVICE)
        eval_global_meta = MetaFusionNet().to(DEVICE)
        
        eval_global_dnn.load_state_dict(global_dnn_sd)
        eval_global_img.load_state_dict(global_img_sd)
        eval_global_meta.load_state_dict(global_meta_sd)
        
        g_acc, g_f1, g_rec, g_auc = evaluate_model(eval_global_dnn, eval_global_img, eval_global_meta, {'main': main_test_loader})
        print(f"\n>>> Round {fl_round} GLOBAL (Main Data) Test Acc: {g_acc:.4f} | Recall: {g_rec:.4f} | F1: {g_f1:.4f} | AUC: {g_auc:.4f} <<<\n")

    torch.save(global_dnn_sd, OUTPUT_DIR / "fl_global_dnn.pth")
    torch.save(global_img_sd, OUTPUT_DIR / "fl_global_resnet.pth")
    torch.save(global_meta_sd, OUTPUT_DIR / "fl_global_meta.pth")
    print(f"Saved federated weights to {OUTPUT_DIR}")

if __name__ == "__main__":
    main()